In [136]:
%load_ext autoreload
%autoreload 2

# Célula 1: Configuração do ambiente e importação da nossa arquitetura
import sys
import os

# Garantir que o notebook enxerga a pasta 'src' na raiz do projeto
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np

# Importar as nossas classes do PEDS
from src.objects import CSstruct, NNstruct, ALstruct
from src.models.surrogate import PEDSModel

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# Testar a instanciação do motor
cs = CSstruct()
nn_params = NNstruct()
modelo = PEDSModel(nn_params)

print("Arquitetura PEDS carregada com sucesso!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PyTorch Version: 2.1.0
CUDA Available: True
Arquitetura PEDS carregada com sucesso!


[autoreload of src.physics.differentiable_physics failed: Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/opt/conda/lib/python3.10/site-packages/IPython/ext

In [137]:
# Célula 2: Pipeline de Ingestão de Dados Originais
import os
import torch
import pandas as pd
import numpy as np

# Mapear o caminho da pasta 'data' na raiz do projeto
data_dir = os.path.abspath(os.path.join('..', 'data'))
x_path = os.path.join(data_dir, "X_maxwell10_small.csv")
y_path = os.path.join(data_dir, "y_maxwell10_small.csv")

print("[ETL] A iniciar a ingestão dos ficheiros CSV...")

# 1. Carregar e transformar a Matriz X (Features)
# O Julia guardou como (Features, Samples). O PyTorch exige (Samples, Features).
X_df = pd.read_csv(x_path, header=None)
X_np = X_df.values.T  # Transposição arquitetural da matriz
X_tensor = torch.tensor(X_np, dtype=torch.float32)

# 2. Carregar e tratar o Vetor Y (Respostas Complexas)
def parse_julia_complex(val):
    if pd.isna(val): return 0j
    # Remove espaços vazios e substitui a sintaxe 'im' do Julia pelo 'j' do Python
    return complex(str(val).replace(' ', '').replace('im', 'j'))

y_df = pd.read_csv(y_path, header=None)
y_flat = y_df.values.flatten() # Garantir que é um vetor 1D
y_np = np.array([parse_julia_complex(v) for v in y_flat])
y_tensor = torch.tensor(y_np, dtype=torch.complex64)

print(f"\n[ETL] Ingestão Concluída!")
print(f"Dataset X Global: {X_tensor.shape}")
print(f"Dataset y Global: {y_tensor.shape}")

# 3. Particionamento Exato dos Dados (Conforme o example_maxwell.jl)
# Nota: O Julia usa índices a começar em 1. O Python começa em 0.
# Xv = X[:, 1:1024] -> Python: [:1024]
X_valid = X_tensor[:1024]
y_valid = y_tensor[:1024]

# Xtest = X[:, end-1023:end] -> Python: [-1024:]
X_test = X_tensor[-1024:]
y_test = y_tensor[-1024:]

# Xt = X[:, 1025:end] -> Python: [1024:]
X_train = X_tensor[1024:]
y_train = y_tensor[1024:]

print("\n[Particionamento] Divisão para o ciclo de Active Learning:")
print(f" - Treino (Xt):     {X_train.shape}")
print(f" - Validação (Xv):  {X_valid.shape}")
print(f" - Teste (Xtest):   {X_test.shape}")

[ETL] A iniciar a ingestão dos ficheiros CSV...

[ETL] Ingestão Concluída!
Dataset X Global: torch.Size([10000, 13])
Dataset y Global: torch.Size([10000])

[Particionamento] Divisão para o ciclo de Active Learning:
 - Treino (Xt):     torch.Size([8976, 13])
 - Validação (Xv):  torch.Size([1024, 13])
 - Teste (Xtest):   torch.Size([1024, 13])


In [138]:
# Célula 4: Treino do Modelo Baseline (Sem Física)
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

# Importar as nossas estruturas
from src.objects import NNstruct
from src.models.surrogate import BaselineModel
from src.models.losses import calculate_nll_loss

print("--- A iniciar o Treino do Modelo Baseline ---")

# 1. Definir a Métrica de Erro Fracionário (Fractional Error - FE)
# Equivalente à função dFE() do Julia
def calculate_fe(rp, ip, y_true):
    y_pred = torch.complex(rp, ip)
    # FE = norm(y_pred - y_true) / norm(y_true)
    return torch.linalg.norm(y_pred - y_true) / torch.linalg.norm(y_true)

# 2. Configurar o Hardware e Subconjunto de Treino (como no artigo: 1280 amostras)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_init = 1280 
batch_size = 64

train_dataset = TensorDataset(X_train[:n_init], y_train[:n_init])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 3. Inicializar o Modelo e Otimizador
nn_params = NNstruct()
baseline = BaselineModel(nn_params).to(device)
optimizer = Adam(baseline.parameters(), lr=1e-3)

# 4. Loop de Treino (Equivalente ao Flux.@epochs do Julia)
epochs = 10
baseline.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        
        # Forward Pass do Baseline (mgen -> pred / mvar)
        z = baseline.mgen(X_batch)
        pred_out = baseline.pred(z) # Devolve (Real, Imaginário)
        vp_out = baseline.mvar(z)   # Devolve (Variância)
        
        # Extrair componentes
        rp = pred_out[:, 0]
        ip = pred_out[:, 1]
        vp = torch.abs(vp_out.squeeze()) + 1e-6 # Garantir variância estritamente positiva
        
        # Calcular Perda e Retropropagar
        loss = calculate_nll_loss((rp, ip, vp), y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Época {epoch+1}/{epochs} | Loss NLL: {epoch_loss/len(train_loader):.4f}")

# 5. Avaliação no Conjunto de Teste (Cálculo do FE)
baseline.eval()
with torch.no_grad():
    X_test_gpu = X_test.to(device)
    y_test_gpu = y_test.to(device)
    
    z_test = baseline.mgen(X_test_gpu)
    pred_test = baseline.pred(z_test)
    
    rp_test = pred_test[:, 0]
    ip_test = pred_test[:, 1]
    
    fe_score = calculate_fe(rp_test, ip_test, y_test_gpu)

print("\n========================================")
print(f"Resultado Baseline (Fractional Error): {fe_score.item():.4f}")
print("========================================")
print("No paper original, o FE do Baseline rondava os 0.541.")

--- A iniciar o Treino do Modelo Baseline ---
Época 1/10 | Loss NLL: 0.4615
Época 2/10 | Loss NLL: 0.2844
Época 3/10 | Loss NLL: 0.0841
Época 4/10 | Loss NLL: -0.1044
Época 5/10 | Loss NLL: -0.1940
Época 6/10 | Loss NLL: -0.1898
Época 7/10 | Loss NLL: -0.4390
Época 8/10 | Loss NLL: -0.4570
Época 9/10 | Loss NLL: -0.5766
Época 10/10 | Loss NLL: -0.6664

Resultado Baseline (Fractional Error): 0.5058
No paper original, o FE do Baseline rondava os 0.541.


In [ ]:
# Célula 5: O Modelo PEDS (Maxwell real, diferenciável)
import torch
import numpy as np
from torch.optim import Adam

from src.objects import NNstruct, CSstruct
from src.models.surrogate import PEDSModel
from src.physics.geometry import SimulationDomain, epsilon_hole_layers
from src.physics.fdfd import FDFDSolver
from src.physics.differentiable_physics import DifferentiableMaxwell
from src.models.losses import calculate_nll_loss

print("--- A inicializar ambiente PEDS (Maxwell) ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Domínio físico do solver (consistente com CSstruct do Maxwell).
#    nx = round(Lx*res) = 10 ; ny = round((Ly+2*dpml)*res) = 210.
cs = CSstruct()
OMEGA = 2 * np.pi  # AJUSTE: confirmar no coarse.jl o mapeamento one-hot -> omega.
sd = SimulationDomain(Lx=cs.Lx, Ly=cs.Ly, omega=OMEGA, dpml=cs.dpml,
                      resolution=cs.resolution, source=cs.source, monitor=cs.monitor)
solver = FDFDSolver(sd)
maxwell = DifferentiableMaxwell(solver)
NY, NX = sd.ny, sd.nx
print(f"Grid do solver: ny={NY}, nx={NX}  (geometria combinada: {NY*NX} pixels)")

# 2. NNstruct configurado para Maxwell (saida no grid completo do solver).
nn_params = NNstruct(
    inGen=[13, 256, 256],
    outGen=[256, 256, NY * NX],
    postGen=[lambda x: x * 1.5 + 2.5,
             lambda x: x.reshape(-1, NY, NX)],
    inVar=[NY * NX, 256, 256, 256],
    outVar=[256, 256, 256, 1],
)
peds_model = PEDSModel(nn_params).to(device)
optimizer_peds = Adam(peds_model.parameters(), lr=1e-3)

# 3. Geometria fisica de baixa fidelidade (prior) a partir das larguras dos furos.
x_grid = sd.xs
y_grid = sd.ys
delta = float(x_grid[1] - x_grid[0])
Lx_grid = float(x_grid[-1] - x_grid[0] + delta)

def build_coarse(X_batch):
    widths = X_batch[:, :10].detach().cpu().numpy().astype(np.float64)
    widths = np.clip(widths, delta + 1e-3, Lx_grid - 1e-3)
    eps_list = []
    for ps in widths:
        geom = epsilon_hole_layers(
            x_grid, y_grid, ps,
            refractive_indexes=tuple(cs.refracsim),
            interstice=cs.interstice, hole=cs.hole,
        )
        eps_list.append(np.real(geom))
    arr = np.stack(eps_list, axis=0)
    return torch.tensor(arr, dtype=torch.float32, device=X_batch.device)

# 4. Forward hibrido: NN + fisica combinadas, solver = PREDICAO.
def physics_informed_forward(X_batch, model):
    coarse = build_coarse(X_batch)
    eps, vp = model(X_batch, coarse)
    rp, ip = maxwell(eps)
    return rp, ip, vp

# 5. Loop de treino
print("--- A iniciar o Treino do Modelo PEDS ---")
peds_model.train()
for epoch in range(10):
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_peds.zero_grad()

        rp, ip, vp_out = physics_informed_forward(X_batch, peds_model)
        vp = torch.abs(vp_out.squeeze()) + 1e-6

        loss = calculate_nll_loss((rp, ip, vp), y_batch)
        loss.backward()
        optimizer_peds.step()
        epoch_loss += loss.item()

    print(f"Epoca {epoch+1} concluida | Loss: {epoch_loss/len(train_loader):.4f}")


In [ ]:
# Célula 17: Motor de Benchmarking (PEDS vs Baseline) - TREINO REAL
from src.models.losses import calculate_nll_loss

def run_experiment(model_type='PEDS'):
    print(f"--- Iniciando Experimento: {model_type} ---")
    
    # 1. Escolher o modelo
    if model_type == 'PEDS':
        model = PEDSModel(nn_params).to(device)
    else:
        model = BaselineModel(nn_params).to(device)
        
    optimizer = Adam(model.parameters(), lr=1e-3)
    model.train()
    
    # 2. Loop de treino real
    for epoch in range(10): 
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            if model_type == 'PEDS':
                # Aqui o 'physics_informed_forward' fará a mágica
                rp, ip, vp_out = physics_informed_forward(X_batch, model)
            else:
                z = model.mgen(X_batch)
                pred_out = model.pred(z)
                rp, ip = pred_out[:, 0], pred_out[:, 1]
                vp_out = model.mvar(z)
                
            vp = torch.abs(vp_out.squeeze()) + 1e-6
            loss = calculate_nll_loss((rp, ip, vp), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        print(f"Época {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")
        
    # 3. Calcular o FE final
    model.eval()
    with torch.no_grad():
        X_test_gpu, y_test_gpu = X_test.to(device), y_test.to(device)
        # Replicar a mesma lógica de forward para o teste
        if model_type == 'PEDS':
            rp, ip, _ = physics_informed_forward(X_test_gpu, model)
        else:
            z = model.mgen(X_test_gpu)
            pred_out = model.pred(z)
            rp, ip = pred_out[:, 0], pred_out[:, 1]
            
        return calculate_fe(rp, ip, y_test_gpu).item()

# Executar o Benchmark
res_peds = run_experiment('PEDS')
res_baseline = run_experiment('Baseline')

print(f"\nResultados Finais:")
print(f"PEDS FE: {res_peds:.4f}")
print(f"Baseline FE: {res_baseline:.4f}")

--- Iniciando Experimento: PEDS ---


NameError: name 'laplacian' is not defined